In [34]:
import requests
import time

# Replace with your personal access token


def github_request(url, params=None):
    """
    A helper function to perform GET requests to the GitHub API,
    returning both the JSON data and the response headers.
    Prints error details if any HTTPError occurs.
    """
    headers = {
       
    }
    try:
        response = requests.get(url, headers=headers, params=params)
        response.raise_for_status()
        return response.json(), response.headers
    except requests.exceptions.HTTPError as e:
        print(f"HTTP error: {e}")
        if e.response is not None:
            print(f"Response text: {e.response.text}")
        raise  # Re-raise so caller knows we failed.

def fetch_repositories(query, max_pages=10, per_page=100):
    """
    Fetch up to max_pages of repositories (max 1,000 total) matching the query,
    because GitHub caps search results at 1,000 items.
    """
    url = "https://api.github.com/search/repositories"
    all_items = []
    
    for page in range(1, max_pages + 1):
        params = {
            "q": query,
            "sort": "stars",   # or "forks" if you prefer
            "order": "desc",
            "per_page": per_page,
            "page": page
        }
        data, headers = github_request(url, params=params)
        
        # "items" holds the array of repos
        items = data.get("items", [])
        if not items:
            break
        
        all_items.extend(items)
        
        # If fewer than per_page returned, no more pages available
        if len(items) < per_page:
            break
        
        # Sleep briefly to avoid secondary rate-limits (abuse detection)
        time.sleep(1)
    
    return all_items

def main():
    # 1. Define your query for large repositories
    #    e.g. "stars:>10000 forks:>10000"
    repo_query = "stars:>10000 forks:>10000"
    
    # 2. Fetch the repositories (up to 1,000 results total)
    repos = fetch_repositories(repo_query, max_pages=10, per_page=100)
    print(f"Found {len(repos)} repositories with >10k stars and forks.\n")
    
    # 3. Print the full_name (owner/repo) for each
    for repo in repos:
        full_name = repo.get("full_name", "unknown/unknown")
        stars = repo.get("stargazers_count", 0)
        forks = repo.get("forks_count", 0)
        print(f"{full_name} - Stars: {stars}, Forks: {forks}")

if __name__ == "__main__":
    main()

Found 254 repositories with >10k stars and forks.

freeCodeCamp/freeCodeCamp - Stars: 408359, Forks: 38549
EbookFoundation/free-programming-books - Stars: 345956, Forks: 62366
sindresorhus/awesome - Stars: 340289, Forks: 28198
public-apis/public-apis - Stars: 322995, Forks: 34268
codecrafters-io/build-your-own-x - Stars: 322412, Forks: 29888
jwasham/coding-interview-university - Stars: 309280, Forks: 77392
kamranahmedse/developer-roadmap - Stars: 303676, Forks: 39768
donnemartin/system-design-primer - Stars: 284377, Forks: 47489
996icu/996.ICU - Stars: 270062, Forks: 21105
facebook/react - Stars: 231036, Forks: 47303
vinta/awesome-python - Stars: 229480, Forks: 25078
practical-tutorials/project-based-learning - Stars: 210211, Forks: 27459
vuejs/vue - Stars: 208238, Forks: 33690
TheAlgorithms/Python - Stars: 196147, Forks: 46058
trekhleb/javascript-algorithms - Stars: 189189, Forks: 30370
tensorflow/tensorflow - Stars: 187088, Forks: 74390
torvalds/linux - Stars: 185323, Forks: 54617
ge

In [1]:
import requests
import time

# Replace with your personal access token


def github_request(url, params=None):
    """
    A helper function to perform GET requests to the GitHub API,
    returning both the JSON data and the response headers.
    If an HTTP error occurs, it prints the response text for debugging.
    """
    headers = {
       
    }
    try:
        response = requests.get(url, headers=headers, params=params)
        response.raise_for_status()
        return response.json(), response.headers
    except requests.exceptions.HTTPError as e:
        print(f"HTTP error: {e}")
        if e.response is not None:
            print(f"Response text: {e.response.text}")
        raise

def fetch_repositories(query, max_pages=10, per_page=100):
    """
    Fetch up to max_pages pages of repositories matching 'query'.
    Each page can have up to 'per_page' items (max 100).
    Returns a list of repo items (dicts).
    """
    url = "https://api.github.com/search/repositories"
    all_items = []
    
    for page in range(1, max_pages + 1):
        params = {
            "q": query,
            "sort": "stars",
            "order": "desc",
            "per_page": per_page,
            "page": page
        }
        data, headers = github_request(url, params=params)
        items = data.get("items", [])
        
        if not items:
            break
        
        all_items.extend(items)
        
        # If fewer than 'per_page', no more pages
        if len(items) < per_page:
            break
        
        # Small delay helps avoid secondary rate limiting
        time.sleep(1)
    
    return all_items

def search_pull_requests_in_repo(owner, repo, pr_query="translate readme in:title is:pr", max_pages=3, per_page=30):
    """
    Searches for pull requests in a specific repo, matching pr_query (default: 'translate readme in:title is:pr').
    Returns a list of PRs (each is a dict with keys like 'title', 'html_url', etc.).
    """
    url = "https://api.github.com/search/issues"
    all_prs = []
    
    for page in range(1, max_pages + 1):
        params = {
            "q": f"repo:{owner}/{repo} {pr_query}",
            "sort": "created",
            "order": "desc",
            "per_page": per_page,
            "page": page
        }
        data, headers = github_request(url, params=params)
        
        items = data.get("items", [])
        if not items:
            break
        
        all_prs.extend(items)
        
        if len(items) < per_page:
            break
        
        time.sleep(1)
    
    return all_prs

def main():
    # 1. Get repositories with >10k stars & forks (adjust as needed)
    repo_query = "stars:>10000 forks:>10000"
    repos = fetch_repositories(repo_query, max_pages=5, per_page=100)
    
    total_repos = len(repos)
    print(f"Found {total_repos} total repos matching {repo_query}.\n")

    # 2. We'll keep a global total of all PRs found
    global_pr_count = 0

    # 3. Process all repositories in chunks/batches of 10
    batch_size = 10
    for start_index in range(0, total_repos, batch_size):
        batch = repos[start_index : start_index + batch_size]
        batch_end_index = start_index + len(batch)

        print(f"Processing repos {start_index + 1} through {batch_end_index} (of {total_repos})...")

        # 4. For each repo in the current batch, search for "translate readme" PRs
        for i, repo_info in enumerate(batch, start=start_index + 1):
            full_name = repo_info.get("full_name")
            if not full_name:
                continue
            
            owner, repo_name = full_name.split("/")
            print(f"{i}. Checking repo '{full_name}' for PRs containing 'translate readme'...")

            prs = search_pull_requests_in_repo(owner, repo_name, max_pages=2, per_page=30)
            pr_count = len(prs)  # number of PRs found in this repo
            global_pr_count += pr_count  # add to our global total

            if pr_count > 0:
                print(f"   Found {pr_count} matching PR(s):")
                for pr in prs:
                    title = pr.get("title")
                    pr_url = pr.get("html_url")
                    print(f"      - {title} ({pr_url})")
            else:
                print("   No matching PRs found.")
            
            print("-" * 60)
            # Optional small sleep between each repo
            time.sleep(2)

        # Optional: sleep after each batch of 10
        print("Finished this batch.\n")
        time.sleep(5)

    # 5. After all batches are processed, print the grand total of PRs
    print(f"Overall total 'translate readme' PRs found across all large repos: {global_pr_count}")

if __name__ == "__main__":
    main()

Found 263 total repos matching stars:>10000 forks:>10000.

Processing repos 1 through 10 (of 263)...
1. Checking repo 'freeCodeCamp/freeCodeCamp' for PRs containing 'translate readme'...
   Found 30 matching PR(s):
      - docs(i18n): reviewing current italian translation of README.md (https://github.com/freeCodeCamp/freeCodeCamp/pull/39388)
      - docs(i18n): translating README.md into Ukrainian (https://github.com/freeCodeCamp/freeCodeCamp/pull/37230)
      - docs(i18n): Vietnamese translation for README.md (https://github.com/freeCodeCamp/freeCodeCamp/pull/37013)
      - docs(i18n): Portuguese translation of README file (https://github.com/freeCodeCamp/freeCodeCamp/pull/36993)
      - docs: add Japanese translation of README.md (https://github.com/freeCodeCamp/freeCodeCamp/pull/36772)
      - doc - Update French README (add links, update translation) (https://github.com/freeCodeCamp/freeCodeCamp/pull/36282)
      - docs: fixed table, references and translations for the Russian vers

Overall total 'translate readme' PRs found across all large repos: 436


In [6]:
import requests
import time

# Replace with your personal access token


def github_request(url, params=None):
    """
    A helper function to perform GET requests to the GitHub API,
    returning both the JSON data and the response headers.
    If an HTTP error occurs, it prints the response text for debugging.
    """
    headers = {
        
    }
    try:
        response = requests.get(url, headers=headers, params=params)
        response.raise_for_status()
        return response.json(), response.headers
    except requests.exceptions.HTTPError as e:
        print(f"HTTP error: {e}")
        if e.response is not None:
            print(f"Response text: {e.response.text}")
        raise

def fetch_repositories(query, max_pages=10, per_page=100):
    """
    Fetch up to max_pages pages of repositories matching 'query'.
    Each page can have up to 'per_page' items (max 100).
    Returns a list of repo items (dicts).
    """
    url = "https://api.github.com/search/repositories"
    all_items = []
    
    for page in range(1, max_pages + 1):
        params = {
            "q": query,
            "sort": "stars",
            "order": "desc",
            "per_page": per_page,
            "page": page
        }
        data, headers = github_request(url, params=params)
        items = data.get("items", [])
        
        if not items:
            break
        
        all_items.extend(items)
        
        # If fewer than 'per_page', no more pages
        if len(items) < per_page:
            break
        
        # Small delay helps avoid secondary rate limiting
        time.sleep(1)
    
    return all_items

def search_pull_requests_in_repo(owner, repo, pr_query="translate readme in:title is:pr", max_pages=3, per_page=30):
    """
    Searches for pull requests in a specific repo, matching pr_query (default: 'translate readme in:title is:pr').
    Returns a list of PRs (each is a dict with keys like 'title', 'html_url', etc.).
    """
    url = "https://api.github.com/search/issues"
    all_prs = []
    
    for page in range(1, max_pages + 1):
        params = {
            "q": f"repo:{owner}/{repo} {pr_query}",
            "sort": "created",
            "order": "desc",
            "per_page": per_page,
            "page": page
        }
        data, headers = github_request(url, params=params)
        
        items = data.get("items", [])
        if not items:
            break
        
        all_prs.extend(items)
        
        if len(items) < per_page:
            break
        
        time.sleep(1)
    
    return all_prs

def main():
    # 1. Get repositories with >10k stars & forks (adjust as needed)
    repo_query = "stars:5000..10000 forks:5000..10000"
    repos = fetch_repositories(repo_query, max_pages=5, per_page=100)
    
    total_repos = len(repos)
    print(f"Found {total_repos} total repos matching {repo_query}.\n")

    # 2. We'll keep a global total of all PRs found
    global_pr_count = 0

    # 3. Process all repositories in chunks/batches of 10
    batch_size = 10
    for start_index in range(0, total_repos, batch_size):
        batch = repos[start_index : start_index + batch_size]
        batch_end_index = start_index + len(batch)

        print(f"Processing repos {start_index + 1} through {batch_end_index} (of {total_repos})...")

        # 4. For each repo in the current batch, search for "translate readme" PRs
        for i, repo_info in enumerate(batch, start=start_index + 1):
            full_name = repo_info.get("full_name")
            if not full_name:
                continue
            
            owner, repo_name = full_name.split("/")
            print(f"{i}. Checking repo '{full_name}' for PRs containing 'translate readme'...")

            prs = search_pull_requests_in_repo(owner, repo_name, max_pages=2, per_page=30)
            pr_count = len(prs)  # number of PRs found in this repo
            global_pr_count += pr_count  # add to our global total

            if pr_count > 0:
                print(f"   Found {pr_count} matching PR(s):")
                for pr in prs:
                    title = pr.get("title")
                    pr_url = pr.get("html_url")
                    print(f"      - {title} ({pr_url})")
            else:
                print("   No matching PRs found.")
            
            print("-" * 60)
            # Optional small sleep between each repo
            time.sleep(2)

        # Optional: sleep after each batch of 10
        print("Finished this batch.\n")
        time.sleep(5)

    # 5. After all batches are processed, print the grand total of PRs
    print(f"Overall total 'translate readme' PRs found across all large repos: {global_pr_count}")

if __name__ == "__main__":
    main()

Found 58 total repos matching stars:5000..10000 forks:5000..10000.

Processing repos 1 through 10 (of 58)...
1. Checking repo 'Yorko/mlcourse.ai' for PRs containing 'translate readme'...
   Found 4 matching PR(s):
      - README: add cn translation link for topic 9 part 2 (https://github.com/Yorko/mlcourse.ai/pull/418)
      - README add link to Chinese translation of topic 10 (https://github.com/Yorko/mlcourse.ai/pull/405)
      - README: add link to cn translation of topic 3. (https://github.com/Yorko/mlcourse.ai/pull/204)
      - README: add link to cn translation of topic 2. (https://github.com/Yorko/mlcourse.ai/pull/191)
------------------------------------------------------------
2. Checking repo 'Klipper3d/klipper' for PRs containing 'translate readme'...
   No matching PRs found.
------------------------------------------------------------
3. Checking repo 'TrinityCore/TrinityCore' for PRs containing 'translate readme'...
   No matching PRs found.
------------------------------

In [7]:
import requests
import time

# Replace with your personal access token


def github_request(url, params=None):
    """
    A helper function to perform GET requests to the GitHub API,
    returning both the JSON data and the response headers.
    If an HTTP error occurs, it prints the response text for debugging.
    """
    headers = {
       
    try:
        response = requests.get(url, headers=headers, params=params)
        response.raise_for_status()
        return response.json(), response.headers
    except requests.exceptions.HTTPError as e:
        print(f"HTTP error: {e}")
        if e.response is not None:
            print(f"Response text: {e.response.text}")
        raise

def fetch_repositories(query, max_pages=10, per_page=100):
    """
    Fetch up to max_pages pages of repositories matching 'query'.
    Each page can have up to 'per_page' items (max 100).
    Returns a list of repo items (dicts).
    """
    url = "https://api.github.com/search/repositories"
    all_items = []
    
    for page in range(1, max_pages + 1):
        params = {
            "q": query,
            "sort": "stars",
            "order": "desc",
            "per_page": per_page,
            "page": page
        }
        data, headers = github_request(url, params=params)
        items = data.get("items", [])
        
        if not items:
            break
        
        all_items.extend(items)
        
        # If fewer than 'per_page', no more pages
        if len(items) < per_page:
            break
        
        # Small delay helps avoid secondary rate limiting
        time.sleep(1)
    
    return all_items

def search_pull_requests_in_repo(owner, repo, pr_query="translate readme in:title is:pr", max_pages=3, per_page=30):
    """
    Searches for pull requests in a specific repo, matching pr_query (default: 'translate readme in:title is:pr').
    Returns a list of PRs (each is a dict with keys like 'title', 'html_url', etc.).
    """
    url = "https://api.github.com/search/issues"
    all_prs = []
    
    for page in range(1, max_pages + 1):
        params = {
            "q": f"repo:{owner}/{repo} {pr_query}",
            "sort": "created",
            "order": "desc",
            "per_page": per_page,
            "page": page
        }
        data, headers = github_request(url, params=params)
        
        items = data.get("items", [])
        if not items:
            break
        
        all_prs.extend(items)
        
        if len(items) < per_page:
            break
        
        time.sleep(1)
    
    return all_prs

def main():
    # 1. Get repositories with >10k stars & forks (adjust as needed)
    repo_query = "stars:100..5000 forks:100..5000"
    repos = fetch_repositories(repo_query, max_pages=5, per_page=100)
    
    total_repos = len(repos)
    print(f"Found {total_repos} total repos matching {repo_query}.\n")

    # 2. We'll keep a global total of all PRs found
    global_pr_count = 0

    # 3. Process all repositories in chunks/batches of 10
    batch_size = 10
    for start_index in range(0, total_repos, batch_size):
        batch = repos[start_index : start_index + batch_size]
        batch_end_index = start_index + len(batch)

        print(f"Processing repos {start_index + 1} through {batch_end_index} (of {total_repos})...")

        # 4. For each repo in the current batch, search for "translate readme" PRs
        for i, repo_info in enumerate(batch, start=start_index + 1):
            full_name = repo_info.get("full_name")
            if not full_name:
                continue
            
            owner, repo_name = full_name.split("/")
            print(f"{i}. Checking repo '{full_name}' for PRs containing 'translate readme'...")

            prs = search_pull_requests_in_repo(owner, repo_name, max_pages=2, per_page=30)
            pr_count = len(prs)  # number of PRs found in this repo
            global_pr_count += pr_count  # add to our global total

            if pr_count > 0:
                print(f"   Found {pr_count} matching PR(s):")
                for pr in prs:
                    title = pr.get("title")
                    pr_url = pr.get("html_url")
                    print(f"      - {title} ({pr_url})")
            else:
                print("   No matching PRs found.")
            
            print("-" * 60)
            # Optional small sleep between each repo
            time.sleep(2)

        # Optional: sleep after each batch of 10
        print("Finished this batch.\n")
        time.sleep(5)

    # 5. After all batches are processed, print the grand total of PRs
    print(f"Overall total 'translate readme' PRs found across all large repos: {global_pr_count}")

if __name__ == "__main__":
    main()

Found 500 total repos matching stars:100..5000 forks:100..5000.

Processing repos 1 through 10 (of 500)...
1. Checking repo 'cweill/gotests' for PRs containing 'translate readme'...
   No matching PRs found.
------------------------------------------------------------
2. Checking repo 'visit1985/mdp' for PRs containing 'translate readme'...
   No matching PRs found.
------------------------------------------------------------
3. Checking repo 'rafaelpadilla/Object-Detection-Metrics' for PRs containing 'translate readme'...
   No matching PRs found.
------------------------------------------------------------
4. Checking repo 'NotionX/react-notion-x' for PRs containing 'translate readme'...
   No matching PRs found.
------------------------------------------------------------
5. Checking repo 'fpillet/NSLogger' for PRs containing 'translate readme'...
   No matching PRs found.
------------------------------------------------------------
6. Checking repo 'alibaba/UltraViewPager' for PRs 